In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler , OneHotEncoder , TargetEncoder  , PowerTransformer , FunctionTransformer
from sklearn.model_selection import train_test_split 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor 
from sklearn.base import BaseEstimator , TransformerMixin 

In [2]:
pd.set_option('display.max_columns' , None)

In [3]:
df = pd.read_csv('../data/processed/osrm_boosted/processed_osrm_NYC.csv')

In [4]:
df.sample(5)

,vendor_id,passenger_count,store_and_fwd_flag,total_distance,total_travel_time,number_of_steps,pickup_zone,pickup_state,dropoff_zone,dropoff_state,distance_haversine_km,distance_manhattan_km,direction,hour_of_day,day_of_week,month_of_year,is_rush_hour,season_name,trip_duration_min,total_distance_km,total_travel_time_min
1382184,2,1,0,2509.3,226.1,6.0,Manhattan,New York,Manhattan,New York,1.936001,2.737611,-134.335387,18,5,1,0,Winter,7.15,2.5093,3.768333
1054295,2,3,0,5888.4,595.3,14.0,Weehawken,New Jersey,New York City,New York,3.901129,4.419902,171.756379,0,6,1,0,Winter,20.15,5.8884,9.921667
349765,2,2,0,3102.5,413.0,13.0,New York City,New York,New York City,New York,2.308631,3.225092,53.968084,14,2,6,0,Summer,17.05,3.1025,6.883333
1262428,2,2,0,14398.7,915.7,17.0,Long Island City,New York,The Bronx,New York,9.911452,12.593015,71.032024,12,1,2,0,Winter,21.38,14.3987,15.261667
1076622,1,1,0,3640.7,303.5,7.0,Guttenberg,New Jersey,Long Island City,New York,3.106671,3.975708,109.805904,8,5,4,0,Spring,9.38,3.6407,5.058333


In [5]:
X = df.drop(columns="trip_duration_min")
y = df['trip_duration_min']

In [6]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [7]:
class NYCTaxiFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that extracts advanced spatial vectors,
    spatio-temporal intersections, and temporal window flags.
    """
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # 1. Advanced Spatio-Temporal Ensembles
        route_comb = X['pickup_zone'].astype(str) + ' -> ' + X['dropoff_zone'].astype(str)
        time_bin = pd.cut(
            X['hour_of_day'], 
            bins=[0, 6, 12, 16, 20, 24], 
            labels=['Night', 'Morning', 'Midday', 'Evening_Rush', 'Late_Night'], 
            include_lowest=True
        ).astype(str)
        
        X['route_time_density'] = route_comb + " @ " + time_bin

        # 2. Advanced Spatial Transformations
        X['is_interstate_trip'] = (X['pickup_state'].astype(str) != X['dropoff_state'].astype(str)).astype(int)

        X['travel_quadrant'] = pd.cut(
            X['direction'], 
            bins=[-180, -90, 0, 90, 180], 
            labels=['South-West', 'North-West', 'North-East', 'South-East'],
            include_lowest=True
        ).astype(str)

        # 3. Advanced Temporal Transformation Contexts
        X['is_late_night'] = X['hour_of_day'].between(0, 5).astype(int)
        X['is_weekend_night'] = (X['is_late_night'] & (X['day_of_week'].isin([4, 5, 6]))).astype(int)

        # Explicitly remove temporary intermediate column states if we do not want them saved
        # X = X.drop(columns=['route_combination', 'time_bin'], errors='ignore')

        return X
    
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        
        # Safe structural type check conversion
        input_features_list = list(input_features)
        
        # Structural fields added inside this transformer class
        new_features = [
            "route_time_density", 
            "is_interstate_trip", 
            "travel_quadrant", 
            "is_late_night", 
            "is_weekend_night"
        ]
        
        return np.array(input_features_list + new_features)

In [8]:
target_encoding_features = ["pickup_zone" , "dropoff_zone" , "route_time_density"]

ohe_features = ['dropoff_state' , 'pickup_state' , 'travel_quadrant' , 'season_name']

numerical_col = [
    'vendor_id', 'passenger_count', 'store_and_fwd_flag',
    'distance_haversine_km', 'distance_manhattan_km', 'direction',
    'hour_of_day', 'day_of_week', 'month_of_year', 'is_rush_hour',
    'is_interstate_trip', 'is_late_night', 'is_weekend_night',
    'total_distance_km' , 'total_travel_time_min'
]

log_transform_features = [
    'distance_haversine_km', 'distance_manhattan_km',
    'total_distance_km' , 'total_travel_time_min'
]

In [9]:
column_transformer = ColumnTransformer(
    transformers=[
        ("target_encoded" , TargetEncoder(smooth="auto") , target_encoding_features),
        ("Ohe" , OneHotEncoder(handle_unknown="ignore" , sparse_output=False) , ohe_features),
        ("log_transform" , FunctionTransformer(np.log1p , validate=True , feature_names_out='one-to-one') , log_transform_features),
        ("std_scalar" , StandardScaler() , numerical_col)
    ],remainder="passthrough"
)

In [10]:
pipeline = Pipeline(
    steps=[
        ("feature_engineering" , NYCTaxiFeatureEngineer()),
        ("column_transformer" , column_transformer)
    ]
)

In [11]:
pipeline.fit(X_train , y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineering', ...), ('column_transformer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('target_encoded', ...), ('Ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold

In [12]:
pipeline.get_feature_names_out()

array(['target_encoded__pickup_zone', 'target_encoded__dropoff_zone',
       'target_encoded__route_time_density',
       'Ohe__dropoff_state_Connecticut', 'Ohe__dropoff_state_New Jersey',
       'Ohe__dropoff_state_New York', 'Ohe__pickup_state_New Jersey',
       'Ohe__pickup_state_New York', 'Ohe__travel_quadrant_North-East',
       'Ohe__travel_quadrant_North-West',
       'Ohe__travel_quadrant_South-East',
       'Ohe__travel_quadrant_South-West', 'Ohe__season_name_Spring',
       'Ohe__season_name_Summer', 'Ohe__season_name_Winter',
       'log_transform__distance_haversine_km',
       'log_transform__distance_manhattan_km',
       'log_transform__total_distance_km',
       'log_transform__total_travel_time_min', 'std_scalar__vendor_id',
       'std_scalar__passenger_count', 'std_scalar__store_and_fwd_flag',
       'std_scalar__distance_haversine_km',
       'std_scalar__distance_manhattan_km', 'std_scalar__direction',
       'std_scalar__hour_of_day', 'std_scalar__day_of_week',


In [13]:
train_trans = pipeline.fit_transform(X_train , y_train)

X_train_trans = pd.DataFrame(train_trans , columns=pipeline.get_feature_names_out())

In [14]:
test_trans = pipeline.transform(X_test)

X_test_trans = pd.DataFrame(test_trans , columns=pipeline.get_feature_names_out())

In [15]:
X_train_trans

,target_encoded__pickup_zone,target_encoded__dropoff_zone,target_encoded__route_time_density,Ohe__dropoff_state_Connecticut,Ohe__dropoff_state_New Jersey,Ohe__dropoff_state_New York,Ohe__pickup_state_New Jersey,Ohe__pickup_state_New York,Ohe__travel_quadrant_North-East,Ohe__travel_quadrant_North-West,Ohe__travel_quadrant_South-East,Ohe__travel_quadrant_South-West,Ohe__season_name_Spring,Ohe__season_name_Summer,Ohe__season_name_Winter,log_transform__distance_haversine_km,log_transform__distance_manhattan_km,log_transform__total_distance_km,log_transform__total_travel_time_min,std_scalar__vendor_id,std_scalar__passenger_count,std_scalar__store_and_fwd_flag,std_scalar__distance_haversine_km,std_scalar__distance_manhattan_km,std_scalar__direction,std_scalar__hour_of_day,std_scalar__day_of_week,std_scalar__month_of_year,std_scalar__is_rush_hour,std_scalar__is_interstate_trip,std_scalar__is_late_night,std_scalar__is_weekend_night,std_scalar__total_distance_km,std_scalar__total_travel_time_min,remainder__total_distance,remainder__total_travel_time,remainder__number_of_steps
0,12.614807,12.486049,9.831993,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.894140,1.079704,1.289260,1.580695,-1.071510,-0.506484,-0.074006,-0.529648,-0.500743,-0.104239,-0.409824,-0.536777,-0.308718,-0.657943,-0.504495,-0.363359,-0.288748,-0.401362,-0.540012,2630.1,231.5,8.0
1,13.923982,13.885359,14.117830,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.857820,1.038736,1.136261,1.778336,0.933262,-0.506484,-0.074006,-0.551618,-0.523143,0.756080,1.152943,0.486473,-0.308718,-0.657943,-0.504495,-0.363359,-0.288748,-0.497934,-0.336820,2115.1,295.2,5.0
2,13.000719,13.888012,9.499205,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.891985,1.110139,0.933580,1.047904,-1.071510,0.254432,-0.074006,-0.530973,-0.483498,0.605754,1.465496,1.509724,-0.903655,-0.657943,-0.504495,-0.363359,-0.288748,-0.605101,-0.924069,1543.6,111.1,5.0
3,13.321668,15.762841,17.468478,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.162659,2.474291,2.421576,2.418292,-1.071510,-0.506484,-0.074006,1.044517,1.192001,0.559709,-1.816315,1.509724,0.286218,-0.657943,-0.504495,2.752102,3.463230,1.030064,0.678825,10263.6,613.6,12.0
4,13.923982,13.541551,9.561328,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.744314,0.849764,0.864787,1.417066,0.933262,-0.506484,-0.074006,-0.615355,-0.615377,0.286966,1.152943,-1.048403,-0.903655,-0.657943,-0.504495,-0.363359,-0.288748,-0.636810,-0.680365,1374.5,187.5,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1125727,12.612984,12.478092,8.934487,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.853171,1.035639,1.038508,1.256565,0.933262,-0.506484,-0.074006,-0.554374,-0.524800,-0.109060,0.527836,1.509724,-0.903655,-0.657943,-0.504495,-0.363359,-0.288748,-0.552333,-0.797432,1825.0,150.8,5.0
1125728,12.617072,13.865426,11.822690,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.252947,1.382330,1.805416,2.155245,-1.071510,-0.506484,-0.074006,-0.263782,-0.303523,1.132924,-2.128868,1.509724,0.286218,-0.657943,-0.504495,2.752102,3.463230,0.058510,0.181848,5082.5,457.8,13.0
1125729,13.920831,12.473944,18.682583,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.565233,1.729997,1.994142,2.487958,-1.071510,1.776265,-0.074006,0.059451,0.010490,0.298051,1.309219,-0.025152,0.881155,-0.657943,-0.504495,-0.363359,-0.288748,0.295421,0.833851,6345.9,662.2,10.0
1125730,13.929438,13.554135,9.866816,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.047876,1.074591,1.257637,1.918147,-1.071510,-0.506484,-0.074006,-0.427284,-0.503589,-0.729310,-0.253548,-0.025152,-1.498591,-0.657943,-0.504495,-0.363359,-0.288748,-0.422551,-0.166801,2517.1,348.5,11.0
